In [1]:
import pandas as pd
import numpy as np
import time
from datetime import datetime
import joblib
from geopy.distance import geodesic

# Load model, scaler, and data
model = joblib.load('bus_eta_predictor.joblib')
scaler = joblib.load('standard_scaler.joblib')
df = pd.read_csv('bus_eta_standard_scaled.csv')

# Fix typo in column name (passenger_count vs passenger_count)
if 'passenger_count' not in df.columns and 'passenger_count' in df.columns:
    df.rename(columns={'passenger_count': 'passenger_count'}, inplace=True)

# Get unique stops in order of appearance
stop_sequence = df.sort_values('stop_sequence')['current_stop_name'].unique()

# Enhanced stop location mapping with typical values
stop_locations = {}
for stop in stop_sequence:
    stop_data = df[df['current_stop_name'] == stop].iloc[0]
    stop_locations[stop] = {
        'lat': stop_data['current_lat'],
        'lon': stop_data['current_lon'],
        'next_stop': df[df['current_stop_name'] == stop]['next_stop_name'].values[0],
        'typical_passengers': int(df[df['current_stop_name'] == stop]['passenger_count'].median()),
        'typical_weather': df[df['current_stop_name'] == stop]['weather_condition'].mode()[0]
    }

def simulate_real_bus_movement():
    print("🚌 Starting Enhanced Bus Simulation with Realistic Dynamics...")
    print("------------------------------------------------------------")
    
    # Initialize with first stop
    current_stop = stop_sequence[0]
    next_stop = stop_locations[current_stop]['next_stop']
    
    # Simulation parameters with realistic starting values
    current_speed = 10  # Start with 10 mph instead of 0 to prevent divide-by-zero
    passenger_count = stop_locations[current_stop]['typical_passengers']
    weather = stop_locations[current_stop]['typical_weather']
    distance_between_stops = geodesic(
        (stop_locations[current_stop]['lat'], stop_locations[current_stop]['lon']),
        (stop_locations[next_stop]['lat'], stop_locations[next_stop]['lon'])
    ).miles
    distance_remaining = distance_between_stops
    
    while next_stop in stop_locations:
        now = datetime.now()
        day_of_week = now.weekday()
        is_peak = 7 <= now.hour <= 9 or 16 <= now.hour <= 19
        is_holiday = False
        
        # Calculate interpolated position
        progress_ratio = 1 - (distance_remaining / distance_between_stops)
        current_lat = stop_locations[current_stop]['lat'] + (
            stop_locations[next_stop]['lat'] - stop_locations[current_stop]['lat']) * progress_ratio
        current_lon = stop_locations[current_stop]['lon'] + (
            stop_locations[next_stop]['lon'] - stop_locations[current_stop]['lon']) * progress_ratio
        
        # Create feature vector - ensure all features match training exactly
        features = pd.DataFrame([{
            'current_stop_name': current_stop,
            'next_stop_name': next_stop,
            'day_of_week': day_of_week,
            'is_holiday': int(is_holiday),
            'is_peak_hour': int(is_peak),
            'weather_condition': weather,
            'passenger_count': passenger_count,  # Note: column name must match your data
            'current_speed': current_speed,
            'distance_to_next_stop': distance_remaining,  # Note: column name must match
            'current_lat': current_lat,
            'current_lon': current_lon
        }])
        
        # Ensure all model features are present and in correct order
        for col in model.feature_names_in_:
            if col not in features.columns:
                # Use median for numerical, mode for categorical
                if col in ['current_stop_name', 'next_stop_name']:
                    features[col] = df[col].mode()[0]
                else:
                    features[col] = df[col].median()
        
        # Reorder columns to match training data
        features = features[model.feature_names_in_]
        
        # PREDICTION WITH ROBUST HANDLING
        try:
            # Get scaled prediction
            eta_scaled = model.predict(features.values.reshape(1, -1))[0]
            
            # Inverse transform
            eta = scaler.inverse_transform([[eta_scaled]])[0][0]
            
            # Physics-based validation
            min_physics_eta = (distance_remaining / max(0.1, current_speed)) * 60
            max_physics_eta = (distance_remaining / max(0.1, current_speed/2)) * 60
            
            # Apply constraints - model must respect physics
            eta = max(0.5, min(eta, max_physics_eta))
            
            # Format to 1 decimal place
            eta = round(eta, 1)
            
        except Exception as e:
            print(f"⚠️ Prediction error: {e}")
            # Fallback to physics-based calculation
            eta = max(0.5, (distance_remaining / max(0.1, current_speed)) * 60)
            eta = round(eta, 1)
        
        # Display
        print(f"\n⏰ {now.strftime('%H:%M:%S')}")
        print(f"📍 Current: {current_stop:>15} → Next: {next_stop:>15}")
        print(f"📏 Distance: {distance_remaining:>5.2f} miles | 🚦 Speed: {current_speed:>4.1f} mph")
        print(f"🌤️ Weather: {weather:>2} | 👥 Passengers: {passenger_count:>3}")
        print(f"⏳ Predicted ETA: {eta:>5.1f} minutes")
        
        # ASCII Art Progress
        progress = int(20 * progress_ratio)
        bus_pos = min(progress, 19)
        print(f"\n[{'·'*bus_pos}🚌{'·'*(19-bus_pos)}] {progress*5:>3}%")
        
        time.sleep(2)
        
        # Realistic movement dynamics
        if current_speed < 25:
            # More realistic acceleration curve
            if progress_ratio < 0.2:  # Accelerate faster at start
                acceleration = 2.5
            elif progress_ratio > 0.8:  # Decelerate when approaching stop
                acceleration = -1.2
            else:  # Maintain speed
                acceleration = 0.2 if current_speed < 20 else 0
            current_speed = max(0, min(25, current_speed + acceleration))
        
        # Random traffic effects
        current_speed *= np.random.uniform(0.9, 1.1)
        current_speed = max(5, min(25, current_speed))  # Keep between 5-25 mph
        
        distance_remaining = max(0, distance_remaining - (current_speed/3600 * 2))
        
        # Arrival logic
        if distance_remaining <= 0.01:
            print(f"\n✅ ARRIVED AT {next_stop}!")
            
            # Passenger dynamics
            passengers_leaving = min(passenger_count, np.random.poisson(3))
            passengers_boarding = np.random.poisson(4 if is_peak else 2)
            passenger_count = max(0, passenger_count - passengers_leaving + passengers_boarding)
            
            # Weather changes
            if np.random.random() < 0.15:
                weather = np.random.choice(df['weather_condition'].unique())
                print(f"🌦️ Weather changed to: {weather}")
            
            # Move to next stop
            current_stop = next_stop
            next_stop = stop_locations[current_stop].get('next_stop')
            
            if next_stop:
                distance_between_stops = geodesic(
                    (stop_locations[current_stop]['lat'], stop_locations[current_stop]['lon']),
                    (stop_locations[next_stop]['lat'], stop_locations[next_stop]['lon'])
                ).miles
                distance_remaining = distance_between_stops
                current_speed = 10  # Reset to starting speed
            else:
                print("🏁 End of route reached!")

# Run simulation
simulate_real_bus_movement()

🚌 Starting Enhanced Bus Simulation with Realistic Dynamics...
------------------------------------------------------------
[LightGBM] [Warning] min_data_in_leaf is set=94, min_child_samples=81 will be ignored. Current value: min_data_in_leaf=94
[LightGBM] [Warning] feature_fraction is set=0.9722793276963305, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9722793276963305
[LightGBM] [Warning] lambda_l1 is set=4.905310569171177e-07, reg_alpha=0.0 will be ignored. Current value: lambda_l1=4.905310569171177e-07
[LightGBM] [Warning] lambda_l2 is set=8.094332988586253e-06, reg_lambda=0.0 will be ignored. Current value: lambda_l2=8.094332988586253e-06
[LightGBM] [Warning] bagging_fraction is set=0.7481514265968381, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7481514265968381
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
⚠️ Prediction error: non-broadcastable output operand with shape (1

: 